In [17]:
%cd /content
!rm -rf 18_10123154_12523066_CDUTV
!git clone https://github.com/YoungHyyy/18_10123154_12523066_CDUTV.git
%cd 18_10123154_12523066_CDUTV

import sys, sklearn, numpy, pandas
sys.path.insert(0, "ai-models/src")
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)

/content
Cloning into '18_10123154_12523066_CDUTV'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 86 (delta 26), reused 75 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 3.59 MiB | 29.14 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/18_10123154_12523066_CDUTV
sklearn 1.6.1 | numpy 2.1.3 | pandas 2.2.3


In [18]:
import ast, pandas as pd
from preprocess import ROOT

results = pd.read_csv(ROOT / "docs" / "model_comparison.csv")

MODEL_NAME = "Logistic Regression (baseline)"   # đổi thành "SVM (RBF)" nếu bạn chọn SVM

row = results[results["model"] == MODEL_NAME].iloc[0]
best_params = ast.literal_eval(row["best_params"])
print("Model chọn:", MODEL_NAME)
print("Siêu tham số:", best_params)
print("Test F1 lúc đánh giá:", row["test_f1"], "| Test Recall:", row["test_recall"])

Model chọn: Logistic Regression (baseline)
Siêu tham số: {'C': 0.1}
Test F1 lúc đánh giá: 0.988 | Test Recall: 0.9762


In [19]:
from preprocess import load_data, make_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

X, y = load_data()   # dùng cả 569 mẫu, không chia train/test nữa

if MODEL_NAME.startswith("Logistic"):
    clf = LogisticRegression(max_iter=5000, class_weight="balanced",
                             random_state=42, **best_params)
elif MODEL_NAME.startswith("SVM"):
    clf = SVC(kernel="rbf", probability=True, class_weight="balanced",
             random_state=42, **best_params)
else:
    raise ValueError("Thêm nhánh cho model này nếu bạn chọn KNN/Decision Tree/Random Forest")

final_model = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
final_model.fit(X, y)
print("Đã huấn luyện xong trên", X.shape[0], "mẫu")

Đã huấn luyện xong trên 569 mẫu


In [20]:
sample = X.iloc[[0]]
pred = final_model.predict(sample)[0]
proba = final_model.predict_proba(sample)[0, 1]
print("Dự đoán mẫu đầu tiên:", "malignant" if pred == 1 else "benign", "| xác suất ác tính:", round(proba, 4))

Dự đoán mẫu đầu tiên: malignant | xác suất ác tính: 1.0


In [21]:
import joblib
from preprocess import MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "model.joblib"
joblib.dump(final_model, model_path, compress=3)
print("Đã lưu:", model_path, "-", round(model_path.stat().st_size / 1024, 1), "KB")

Đã lưu: /content/18_10123154_12523066_CDUTV/ai-models/models/model.joblib - 2.4 KB


In [22]:
import json, datetime, sklearn, numpy, pandas

metadata = {
    "model_name": MODEL_NAME,
    "model_version": "1.0.0",
    "trained_on": datetime.date.today().isoformat(),
    "hyperparameters": best_params,
    "metrics_at_selection": {          # đo trên tập test 20%, lúc so sánh model
        "test_recall": float(row["test_recall"]),
        "test_precision": float(row["test_precision"]),
        "test_f1": float(row["test_f1"]),
        "test_roc_auc": float(row["test_roc_auc"]),
    },
    "trained_on_full_dataset": True,   # bản lưu này học trên toàn bộ 569 mẫu
    "library_versions": {
        "scikit-learn": sklearn.__version__,
        "numpy": numpy.__version__,
        "pandas": pandas.__version__,
    },
    "positive_class": "malignant",
    "labels": {"0": "benign", "1": "malignant"},
}

with open(MODELS_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

{
  "model_name": "Logistic Regression (baseline)",
  "model_version": "1.0.0",
  "trained_on": "2026-09-25",
  "hyperparameters": {
    "C": 0.1
  },
  "metrics_at_selection": {
    "test_recall": 0.9762,
    "test_precision": 1.0,
    "test_f1": 0.988,
    "test_roc_auc": 0.9977
  },
  "trained_on_full_dataset": true,
  "library_versions": {
    "scikit-learn": "1.6.1",
    "numpy": "2.1.3",
    "pandas": "2.2.3"
  },
  "positive_class": "malignant",
  "labels": {
    "0": "benign",
    "1": "malignant"
  }
}


In [23]:
from google.colab import files
files.download(str(MODELS_DIR / "model.joblib"))
files.download(str(MODELS_DIR / "metadata.json"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Đóng gói model

**Model được chọn:** Logistic Regression, C = 0.1 (xem lý do chọn ở notebook 04).

**Vị trí trong repo:** `ai-models/models/model.joblib`. File chứa cả pipeline tiền xử lý (SimpleImputer + StandardScaler) và model, nên khi dự đoán chỉ cần gọi `model.predict()` trên dữ liệu thô, không cần xử lý tay.

**Cách export từ Colab:** chạy `ai-models/colab/05_package.ipynb`, huấn luyện lại model đã chọn trên toàn bộ 569 mẫu (dùng đúng siêu tham số đã tìm được ở bước tinh chỉnh), rồi tải `model.joblib` và `metadata.json` về máy và commit lên Git. AI Service đọc trực tiếp file này khi container khởi động.

**Phiên bản thư viện lúc huấn luyện:** scikit-learn 1.6.1, numpy 2.1.3 pandas 2.2.3

. Các phiên bản này được ghim trong `ai-models/requirements.txt` để môi trường Docker khớp với lúc huấn luyện.